# example / wasm / accumulate

**Intent**
- Takes `wasm_in` and the state of the previous call.
- Outputs `wasm_out`, the input scaled by a fitted gain plus a fitted offset.
- Outputs `wasm_*`, 28 check outputs, one family each of comparison, rounding, power and host maths.
- Outputs `wasm_state_out`, the running total and peak.
- Exercises every tracer operator and all fifteen host maths functions in all three hosts.
- Acceptance holds the rmse to 0.5, and each check output to 1e-9 against plain Python.
- It is a fixture, tested by `fab unittest` and `fab systest` and never released.

**Data**
- `training` is ten labelled points on a line with fixed noise, the only data the fit sees.
- `validation` is five more points, and `backtest` is ten further out, held back for acceptance.
- `golden` is six calls from 1 to 14 and back to 2, crossing both bounds and the ties of `nearest`.
- `golden` ends with a request with the input missing, which must give a signature error.
- The golden tolerance is 1e-9 for outputs that depend on a host maths function.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from os.path import join

import numpy as np

from workbench import setup_workspace

ws = setup_workspace("example/wasm/accumulate", {"kind": "embedded", "language": "python"})


INPUT = ws.inputs[0]
OUTPUT = ws.outputs[0]

## Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-data))

In [ ]:
from workbench import data_load

NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [float(x) for x in range(first, first + count)]
    return {INPUT: xs, OUTPUT: [round(2.5 * x + 1.0 + NOISE[int(x) % 10], 4) for x in xs]}


training_pl, training_path = data_load(ws, "training", "wasm_accumulate_training", run=1, source="builder", build=lambda: labelled(0, 10))

validation_pl, validation_path = data_load(ws, "validation", "wasm_accumulate_validation", run=1, source="builder", build=lambda: labelled(10, 5))

backtest_pl, backtest_path = data_load(ws, "backtest", "wasm_accumulate_backtest", run=1, source="builder", build=lambda: labelled(15, 10))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Train ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-train))

In [ ]:
gain, offset = np.polyfit(training_pl[INPUT].to_numpy(), training_pl[OUTPUT].to_numpy(), 1)
GAIN, OFFSET = round(float(gain), 1), round(float(offset), 1)
print("fitted gain", GAIN, "offset", OFFSET)
assert (GAIN, OFFSET) == (2.5, 1.0), "the fit did not recover the line that made the data"

## Evaluate ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-evaluate))

In [ ]:
from workbench import evaluate_check, evaluate_measure, evaluate_predict, wasm

LOW, HIGH = 5.0, 20.0
UNARY = ("acos", "asin", "atan", "cos", "cosh", "exp", "log", "log10", "log2", "sin", "sinh", "tan", "tanh")
STATE_IN = "wasm_state_in"
STATE_OUT = "wasm_state_out"


def exercise(x, out):
    unit = wasm.clip(x / 16.0, 0.0, 1.0)
    checks = {
        "bounded": wasm.clip(out, LOW, HIGH),
        "band": wasm.select(out < LOW, -1.0, wasm.select(out > HIGH, 1.0, 0.0)),
        "band_inclusive": wasm.select(out <= LOW, 0.0, wasm.select(out >= HIGH, 2.0, 1.0)),
        "equal": wasm.select(wasm.eq(out, 8.5), 1.0, 0.0),
        "unequal": wasm.select(wasm.ne(out, 8.5), 1.0, 0.0),
        "floor": wasm.floor(out),
        "ceil": wasm.ceil(out),
        "trunc": wasm.trunc(-out),
        "nearest": wasm.nearest(out),
        "magnitude": wasm.sqrt(abs(-out)),
        "polynomial": 0.01 * out**3 - 0.2 * out**2 + out,
        "power_const": out**1.5,
        "power_base": 2.0**unit,
        "power_unit": wasm.power(unit, 2.5),
        "atan2": wasm.atan2(out, x),
    }
    return checks | {name: getattr(wasm, name)(unit) for name in UNARY}


def accumulate(x, total, peak, gain, offset):
    out = gain * x + offset
    return [out, *exercise(x, out).values(), total + out, wasm.maximum(peak, out)]


CHECKS = list(exercise(1.0, 3.5))
TOLERANT = [name for name in CHECKS if name in (*UNARY, "polynomial", "power_const", "power_base", "power_unit", "atan2")]

model_path = join(ws.path, "model.wasm")
with open(model_path, "wb") as model_file:
    model_file.write(wasm.train_wasm_build(3, wasm.train_wasm_trace(lambda x, total, peak: accumulate(x, total, peak, GAIN, OFFSET), 3)))

predict = evaluate_predict(model_path, "wasm", ws.signature)


def acceptance_checks():
    checks = []
    for split, frame in (("validation", validation_pl), ("backtest", backtest_pl)):
        xs = frame[INPUT].to_list()
        predicted = [predict({INPUT: [x]}) for x in xs]
        checks.append(
            {
                "split": split,
                "output": OUTPUT,
                "name": "rmse",
                "value": evaluate_measure(frame[OUTPUT].to_list(), [answer[OUTPUT][0] for answer in predicted])["rmse"],
                "limit": 0.5,
                "direction": "max",
            }
        )
        for name in CHECKS:
            plain = [exercise(x, GAIN * x + OFFSET)[name] for x in xs]
            checks.append(
                {
                    "split": split,
                    "output": f"wasm_{name}",
                    "name": "rmse",
                    "value": evaluate_measure(plain, [answer[f"wasm_{name}"][0] for answer in predicted])["rmse"],
                    "limit": 1e-9,
                    "direction": "max",
                }
            )
    return checks


evaluation = evaluate_check(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "limit", entry["limit"], "passed", entry["passed"])

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench import package_workspace


def golden_samples():
    xs = [1.0, 3.0, 8.0, 10.0, 14.0, 2.0]
    names = [OUTPUT, *[f"wasm_{name}" for name in CHECKS]]
    rows, total, peak = [], 0.0, 0.0
    for index, x in enumerate(xs):
        *scalars, next_total, next_peak = accumulate(x, total, peak, GAIN, OFFSET)
        rows.append({INPUT: x, STATE_IN: None if index == 0 else f"{total!r} {peak!r}", **dict(zip(names, scalars, strict=True)), STATE_OUT: f"{next_total!r} {next_peak!r}", "expect": "ok"})
        total, peak = next_total, next_peak
    rows.append({"expect": "error"})
    return {key: [row.get(key) for row in rows] for key in (INPUT, STATE_IN, *names, STATE_OUT, "expect")}


golden = golden_samples()
package_workspace(
    ws,
    "wasm",
    golden,
    trained_on="synthetic line, training split",
    inputs=[training_path],
    notes="least squares line, gain and offset built into the module",
    implementation="WebAssembly traced from a Python function by workbench.wasm.trace and built by train_wasm_build, importing every asystem_math function",
    model_path=model_path,
    evaluation=evaluation,
    sequence="a running total and peak over six calls",
    tolerance={f"wasm_{name}": {"absolute": 1e-9, "relative": 1e-9} for name in TOLERANT},
)